# Train and evaluate Qwen2.5-VL-3B on CORD v2

Model: `unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit` with QLoRA 4-bit (language/attention/MLP only, r=16, α=32, dropout 0).
Inference: long-edge 1008 / `max_pixels = 1008×28×28`. Training: `TRAIN_VISION_TOKENS=512` so image+JSON fit `TRAIN_MAX_SEQ=3072` on a T4 — do not raise `max_length` to 8192 and do not cut it without shrinking pixels.
Dataset: official CORD v2 splits. Test ids never enter training.
CORD has no store/date labels; those targets are null (F1 0 is expected).

Colab: Runtime → GPU (T4). `import unsloth` before transformers. For the full standalone pipeline use `notebooks/03_full_pipeline.ipynb` or `notebooks/Full_pipeline.ipynb`.

In [1]:
# Install notes (Colab)
# %pip install -q datasets transformers peft trl bitsandbytes accelerate wandb pillow
# %pip install -q unsloth || echo 'unsloth optional'
import os, json, torch
os.environ.setdefault('WANDB_MODE', 'disabled' if not os.environ.get('WANDB_API_KEY') else 'online')
HAS_GPU = torch.cuda.is_available()
print('cuda', HAS_GPU)

ModuleNotFoundError: No module named 'torch'

In [ ]:
from datasets import load_dataset
from pathlib import Path
from training.prepare_cord import cord_to_schema, freeze_splits, ground_truth_of, to_messages

dataset = load_dataset('naver-clova-ix/cord-v2')
splits = freeze_splits(dataset, Path('data/splits.json'))
print({k: len(v) for k, v in splits.items()})
example = cord_to_schema(ground_truth_of(dataset['train'][0]))
print(json.dumps(example, indent=2)[:500])

In [ ]:
import os
try:
    import wandb
    if os.environ.get('WANDB_MODE') != 'disabled':
        wandb.init(project=os.environ.get('WANDB_PROJECT', 'vlm-receipt-extraction'), config={
            'model_id': 'Qwen/Qwen2.5-VL-3B-Instruct', 'lora_r': 16, 'lora_alpha': 32,
            'lora_dropout': 0.0, 'bits': 4, 'train_vision_tokens': 512,
            'train_size': len(dataset['train']), 'val_size': len(dataset['validation']),
            'learning_rate': 5e-5, 'split_hash': json.load(open('data/splits.json')).get('hash'),
        })
except Exception as exc:
    print('wandb disabled', exc)

## Zero-shot eval (needs GPU)
Skip on CPU. On a T4, evaluate a small frozen test subset, then train, then evaluate the same ids.

In [ ]:
if not HAS_GPU:
    print('needs GPU — skip zero-shot/QLoRA cells. Dummy metrics: python scripts/evaluate.py')
else:
    print('Run training/train_qlora.py or continue with Unsloth cells below.')

In [ ]:
# QLoRA (needs GPU). Caps vision tokens, skips NaN grads, restores last finite checkpoint.
if HAS_GPU:
    !python training/train_qlora.py --epochs 1 --max-samples 200 --learning-rate 5e-5 --max-seq 3072 --lora-dropout 0
else:
    print('skipped training')

In [ ]:
# Optional push: HF_PUSH=1 HF_ADAPTER_REPO=your-username/vlm-receipt-extraction-lora
print('Adapter stays local unless HF_PUSH=1')
# Fine-tuned eval would call eval/run_eval.py on the same test ids.
from scripts.evaluate import main
main()